# 第5周｜pandas清洗、分组与合并

**学习目标：**识别缺失和重复，生成汇总表并连接股票信息。

先预测、再运行、最后解释。允许查课件，重点是读懂和修改代码。

**使用方式：**选择课程 `.venv` 内核，从上到下运行。示例可以直接运行；练习单元格中的注释是待补全位置，取消注释并填写。每题按本周准备单元格独立取数，不依赖上一题答案。参考答案默认折叠，先独立完成再展开。

## 准备与小示例

沿用第4周15行合成行情，人为添加1条完全重复行、1个缺失价格和代码两侧空格。info每只股票一行。dirty是本周原始练习表，不要覆盖它。

In [1]:
import pandas as pd

df = pd.DataFrame({
    "date": list(pd.date_range("2025-01-06", periods=5, freq="B")) * 3,
    "code": ["A"] * 5 + ["B"] * 5 + ["C"] * 5,
    "close": [10, 11, 10.5, 12, 11.5, 20, 19, 20, 21, 20.5, 5, 5.1, 5, 5.2, 5.3],
    "volume": [1000, 1200, 900, 1500, 1100, 800, 900, 1000, 1200, 950, 2000, 2100, 1800, 2200, 2300],
})

dirty = pd.concat([df, df.iloc[[0]]], ignore_index=True)
dirty.loc[6, "close"] = float("nan")
dirty["code"] = " " + dirty["code"] + " "
info = pd.DataFrame({"code": ["A", "B", "C"],
                     "name": ["示例股票A", "示例股票B", "示例股票C"],
                     "industry": ["制造", "金融", "制造"]})

In [2]:
word = " A "
print(repr(word), repr(word.strip()))
display(dirty.head(3))

' A ' 'A'


,date,code,close,volume
0,2025-01-06,A,10.0,1000
1,2025-01-07,A,11.0,1200
2,2025-01-08,A,10.5,900


**输出解释：**strip去掉字符串两侧空格；整列字符串操作使用.str.strip()。repr帮助看见空格。

## 练习1｜找出问题

统计每列缺失数及完全重复行数，显示缺失价格的记录。

In [3]:
# print(dirty.isna().sum())
# print(dirty.duplicated().sum())
# display(...)

**我的解释：**（填写预测、实际结果或原因。）

<details><summary>参考答案（先自己动手）</summary>

```python
print(dirty.isna().sum())
print(dirty.duplicated().sum())
display(dirty.loc[dirty["close"].isna()])
```

close有1个缺失，完全重复行有1条。缺失记录存在，但没有可用价格。

</details>

## 练习2｜清理字符串与重复

复制dirty为clean，去除代码空格并删除完全重复行；报告前后行数。

In [4]:
# clean = dirty.copy()
# clean["code"] = ...
# clean = ...

**我的解释：**（填写预测、实际结果或原因。）

<details><summary>参考答案（先自己动手）</summary>

```python
clean = dirty.copy()
clean["code"] = clean["code"].str.strip()
clean = clean.drop_duplicates()
print(len(dirty), len(clean))
print(clean["code"].unique())
```

16行变为15行；这里删除的是完全重复行，不是随意删除同一股票不同日期的记录。

</details>

## 练习3｜缺失值怎么处理

本题仅用于价格描述统计，删除缺失价格行。解释为什么不直接把价格填0。为保证独立运行，提供前两步代码。

In [5]:
clean = dirty.copy()
clean["code"] = clean["code"].str.strip()
clean = clean.drop_duplicates()
# valid = clean.dropna(subset=[...])
# print(len(valid))

**我的解释：**（填写预测、实际结果或原因。）

<details><summary>参考答案（先自己动手）</summary>

```python
clean = dirty.copy()
clean["code"] = clean["code"].str.strip()
clean = clean.drop_duplicates()
valid = clean.dropna(subset=["close"])
print(len(valid))
```

剩余14行；缺失不是零价格。删除后B只有4个价格观测，后续比较需报告样本数；本题不计算跨缺失日期的日收益。

</details>

## 练习4｜按股票汇总

对下方已整理好的valid，按股票计算价格count、mean、max、min，再按mean降序排列。

In [6]:
valid = dirty.assign(code=dirty["code"].str.strip()).drop_duplicates().dropna(subset=["close"])
# summary = valid.groupby(...)[...].agg([...])
# display(summary.sort_values(..., ascending=False))

**我的解释：**（填写预测、实际结果或原因。）

<details><summary>参考答案（先自己动手）</summary>

```python
valid = dirty.assign(code=dirty["code"].str.strip()).drop_duplicates().dropna(subset=["close"])
summary = valid.groupby("code")["close"].agg(["count", "mean", "max", "min"])
display(summary.sort_values("mean", ascending=False))
```

A、B、C有效报价数分别为5、4、5。均价约为11、20.375、5.12元；价格水平不能用于判断收益表现。准备行中的assign生成清理后的code列，后续两个方法依次去重、去缺失。

</details>

## 练习5｜连接股票信息

将valid与info按code左连接。先预测结果行数；解释为什么不会变成3行，以及右表代码重复时可能发生什么。

In [7]:
valid = dirty.assign(code=dirty["code"].str.strip()).drop_duplicates().dropna(subset=["close"])
# merged = valid.merge(info, on=..., how=...)
# display(merged.head())
# print(len(valid), len(merged))

**我的解释：**（填写预测、实际结果或原因。）

<details><summary>参考答案（先自己动手）</summary>

```python
valid = dirty.assign(code=dirty["code"].str.strip()).drop_duplicates().dropna(subset=["close"])
merged = valid.merge(info, on="code", how="left")
display(merged.head())
print(len(valid), len(merged))
```

左右结果都是14行。左表一只股票多个日期，右表每只股票恰好一行，因此只是补充信息；右表代码重复可能让同一左表记录匹配多次。

</details>

## 提交与自查

- 报告16→15→14的行数变化及原因。
- 解释连接键和两张表每行含义。

提交本周 Notebook，保留你的代码、输出和文字解释。重启内核并按顺序运行一次；不要只提交截图。如使用AI辅助，请注明用在哪一题，并确保能解释关键行、现场修改一个参数。